### Use a pre-trained model for image classification with fine-tuning.

In [ ]:
!pip install tensorflow

In [1]:
# Step 1: Import Libraries
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2 # Changed from ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

In [2]:
# Step 1: Load dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
y_train, y_test = to_categorical(y_train), to_categorical(y_test)

# Normalize
x_train, x_test = x_train / 255.0, x_test / 255.0

In [3]:
# Step 2: Create TensorFlow datasets (resizing done on the fly)
def preprocess(image, label):
    image = tf.image.resize(image, (224, 224))
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)).map(preprocess).shuffle(5000).batch(32).prefetch(1)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(preprocess).batch(32).prefetch(1)


In [4]:
# Step 3: Load pre-trained MobileNetV2
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3)) # Changed from ResNet50

# Freeze most layers
for layer in base_model.layers[:-10]:
    layer.trainable = False

In [5]:
# Step 4: Add custom layers
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)
output = Dense(10, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=output)

In [6]:
# Step 5: Compile
model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Step 6: Train (fine-tune)
history = model.fit(train_ds, validation_data=test_ds, epochs=5)

Epoch 1/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 106s 53ms/step - accuracy: 0.6578 - loss: 0.9986 - val_accuracy: 0.7510 - val_loss: 0.9011
Epoch 2/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 67s 41ms/step - accuracy: 0.8060 - loss: 0.5565 - val_accuracy: 0.8163 - val_loss: 0.5476
Epoch 3/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 67s 41ms/step - accuracy: 0.8325 - loss: 0.4848 - val_accuracy: 0.8264 - val_loss: 0.5124
Epoch 4/5


In [7]:
# Step 6 (continued): Train (fine-tune) with checkpointing
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model

# Define the callback
checkpoint_callback = ModelCheckpoint(
    filepath='best_model_checkpoint.keras',  # Filepath to save the model
    save_best_only=True,  # Save only the best model based on validation accuracy
    monitor='val_accuracy',  # Metric to monitor
    mode='max',  # Mode can be 'auto', 'min', or 'max'
    verbose=1 # Verbosity level
)

# To resume training from a checkpoint, first load the model
try:
    model = load_model('best_model_checkpoint.keras')
    print("Checkpoint model loaded successfully.")
    # Determine the initial epoch to resume from
    # This assumes your history object or a similar mechanism tracks the last completed epoch
    # For simplicity here, we'll assume we want to resume from epoch 1 if a checkpoint exists.
    initial_epoch = 3 # Assuming the checkpoint was saved after epoch 0
except Exception as e:
    print(f"Could not load checkpoint model: {e}")
    print("Starting training from scratch.")
    initial_epoch = 0
    # You would typically redefine your model architecture here if starting from scratch
    # However, for this example, we'll assume the model architecture is already defined in previous cells


# Train the model with the callback
# Set initial_epoch to the epoch after the one that was last completed based on your checkpoint.
# For example, if your checkpoint is from the end of epoch 1, set initial_epoch=1 to start epoch 2.
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5, # Total number of epochs you want to train for
    initial_epoch=initial_epoch, # Start from this epoch
    callbacks=[checkpoint_callback] # Add the callback here
)

Checkpoint model loaded successfully.
Epoch 4/5
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.8449 - loss: 0.4421
Epoch 4: val_accuracy improved from -inf to 0.82890, saving model to best_model_checkpoint.keras
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 116s 59ms/step - accuracy: 0.8449 - loss: 0.4421 - val_accuracy: 0.8289 - val_loss: 0.5048
Epoch 5/5
1562/1563 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8592 - loss: 0.4001
Epoch 5: val_accuracy improved from 0.82890 to 0.83750, saving model to best_model_checkpoint.keras
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 71s 43ms/step - accuracy: 0.8592 - loss: 0.4001 - val_accuracy: 0.8375 - val_loss: 0.4866


In [8]:
# Step 7: Evaluate & Save
loss, acc = model.evaluate(test_ds)
print(f"\n✅ Validation Accuracy: {acc*100:.2f}%")

313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 31ms/step - accuracy: 0.8320 - loss: 0.4913

✅ Validation Accuracy: 83.75%
